In [1]:
import pandas as pd
import numpy as np

raw = pd.DataFrame({
    'txn_id': [1, 2, 3, 3, 4, 5, 6, 7, 8],
    'user_id': ['u1', 'u2', 'u1', 'u1', 'u3', 'u2', 'u4', 'u1', 'u3'],
    'category': [' Electronics', 'grocery ', 'ELECTRONICS', 'ELECTRONICS', 'Apparel', 'Grocery', 'apparel', 'Grocery', 'Electronics'],
    'spend_str': ['$1,200.50', '$45.00', '$300', '$300', 'N/A', '$80.25', '$150.00', '$60', '$950'],
    'txn_date': ['2026-01-05', '01/12/2026', '2026-02-01', '2026-02-01', '2026-02-14', '2026-03-02', '03/10/2026', '2026-03-15', '2026-03-20']
})

In [2]:
raw

,txn_id,user_id,category,spend_str,txn_date
0,1,u1,Electronics,"$1,200.50",2026-01-05
1,2,u2,grocery,$45.00,01/12/2026
2,3,u1,ELECTRONICS,$300,2026-02-01
3,3,u1,ELECTRONICS,$300,2026-02-01
4,4,u3,Apparel,N/A,2026-02-14
5,5,u2,Grocery,$80.25,2026-03-02
6,6,u4,apparel,$150.00,03/10/2026
7,7,u1,Grocery,$60,2026-03-15
8,8,u3,Electronics,$950,2026-03-20


##### 1. Clean `raw` into a new DataFrame `clean`

*(Without mutating `raw` or triggering copy/view warnings)*

- Remove duplicate `txn_id` rows (keep the first).
- Standardize `category`:
  - Remove extra spaces
  - Convert to Title Case
- Parse `spend_str` into a numeric `spend` float column:
  - `'N/A'` → `NaN`
- Parse `txn_date` (mixed formats) into a proper `datetime64` column.

In [89]:
# 1
raw_cleaned = raw.drop_duplicates(keep='first', subset=['txn_id']).copy()
raw_cleaned

,txn_id,user_id,category,spend_str,txn_date
0,1,u1,Electronics,"$1,200.50",2026-01-05
1,2,u2,grocery,$45.00,01/12/2026
2,3,u1,ELECTRONICS,$300,2026-02-01
4,4,u3,Apparel,N/A,2026-02-14
5,5,u2,Grocery,$80.25,2026-03-02
6,6,u4,apparel,$150.00,03/10/2026
7,7,u1,Grocery,$60,2026-03-15
8,8,u3,Electronics,$950,2026-03-20


In [90]:
# 2

raw_cleaned['category'] = raw_cleaned['category'].str.title().str.strip()
raw_cleaned

,txn_id,user_id,category,spend_str,txn_date
0,1,u1,Electronics,"$1,200.50",2026-01-05
1,2,u2,Grocery,$45.00,01/12/2026
2,3,u1,Electronics,$300,2026-02-01
4,4,u3,Apparel,N/A,2026-02-14
5,5,u2,Grocery,$80.25,2026-03-02
6,6,u4,Apparel,$150.00,03/10/2026
7,7,u1,Grocery,$60,2026-03-15
8,8,u3,Electronics,$950,2026-03-20


In [91]:
# 3
raw_cleaned['spend_str'] = raw_cleaned['spend_str'].replace('N/A',np.nan)
raw_cleaned

,txn_id,user_id,category,spend_str,txn_date
0,1,u1,Electronics,"$1,200.50",2026-01-05
1,2,u2,Grocery,$45.00,01/12/2026
2,3,u1,Electronics,$300,2026-02-01
4,4,u3,Apparel,NaN,2026-02-14
5,5,u2,Grocery,$80.25,2026-03-02
6,6,u4,Apparel,$150.00,03/10/2026
7,7,u1,Grocery,$60,2026-03-15
8,8,u3,Electronics,$950,2026-03-20


In [92]:
raw_cleaned['spend'] = raw_cleaned['spend_str'].str.extract(r'([0-9|\,\.]+)', expand=False).str.replace(',','').astype(float)
raw_cleaned

,txn_id,user_id,category,spend_str,txn_date,spend
0,1,u1,Electronics,"$1,200.50",2026-01-05,1200.50
1,2,u2,Grocery,$45.00,01/12/2026,45.00
2,3,u1,Electronics,$300,2026-02-01,300.00
4,4,u3,Apparel,NaN,2026-02-14,NaN
5,5,u2,Grocery,$80.25,2026-03-02,80.25
6,6,u4,Apparel,$150.00,03/10/2026,150.00
7,7,u1,Grocery,$60,2026-03-15,60.00
8,8,u3,Electronics,$950,2026-03-20,950.00


In [93]:
# 4
raw_cleaned['txn_date'] = pd.to_datetime(raw_cleaned['txn_date'], errors='coerce').fillna(pd.to_datetime(raw_cleaned['txn_date'], format = '%d/%m/%Y', errors = 'coerce'))
raw_cleaned

,txn_id,user_id,category,spend_str,txn_date,spend
0,1,u1,Electronics,"$1,200.50",2026-01-05,1200.50
1,2,u2,Grocery,$45.00,2026-12-01,45.00
2,3,u1,Electronics,$300,2026-02-01,300.00
4,4,u3,Apparel,NaN,2026-02-14,NaN
5,5,u2,Grocery,$80.25,2026-03-02,80.25
6,6,u4,Apparel,$150.00,2026-10-03,150.00
7,7,u1,Grocery,$60,2026-03-15,60.00
8,8,u3,Electronics,$950,2026-03-20,950.00


##### 2. Spend Tiers & Summary Matrix

*(Using only valid `spend` rows)*

- Bucket each transaction into `spend_tier`:
  - `'Low'` → `0–100`
  - `'Medium'` → `100–500`
  - `'High'` → `> 500`

- Create a summary table with:
  - `category` as rows
  - `spend_tier` as columns
  - Values = **total spend**

- Fill missing combinations with `0`.

- Include:
  - Row totals
  - Column totals

In [94]:
# 1
def tier(x):
    if pd.isna(x) == False:
        if x >=0 and x<=100:
            return 'Low'
        elif x>100 and x<=500:
            return 'Medium'
        else:
            return 'High'
    
raw_cleaned['spend_tier'] = raw_cleaned['spend'].apply(tier)
raw_cleaned

,txn_id,user_id,category,spend_str,txn_date,spend,spend_tier
0,1,u1,Electronics,"$1,200.50",2026-01-05,1200.50,High
1,2,u2,Grocery,$45.00,2026-12-01,45.00,Low
2,3,u1,Electronics,$300,2026-02-01,300.00,Medium
4,4,u3,Apparel,NaN,2026-02-14,NaN,NaN
5,5,u2,Grocery,$80.25,2026-03-02,80.25,Low
6,6,u4,Apparel,$150.00,2026-10-03,150.00,Medium
7,7,u1,Grocery,$60,2026-03-15,60.00,Low
8,8,u3,Electronics,$950,2026-03-20,950.00,High


In [ ]:
pd.cut(raw_cleaned['spend'], bins=[0, 100, 500, np.inf], labels=['Low', 'Medium', 'High'])

In [95]:
# 2

raw_cleaned.pivot_table(index = 'category',columns='spend_tier', values='spend', fill_value=0, aggfunc='sum', margins=True)

spend_tier,High,Low,Medium,All
category,,,,
Apparel,0.0,0.00,150.0,150.00
Electronics,2150.5,0.00,300.0,2450.50
Grocery,0.0,185.25,0.0,185.25
All,2150.5,185.25,450.0,2785.75


##### 3. User Spend Segmentation

- Compute total valid `spend` per `user_id`.

- Assign each user into 2 equal-count buckets:
  - `'Bottom 50%'`
  - `'Top 50%'`

In [99]:
is_grp = raw_cleaned.groupby('user_id')

In [101]:
spend_by_user = pd.DataFrame(is_grp['spend'].sum().sort_values())
spend_by_user

,spend
user_id,
u2,125.25
u4,150.00
u3,950.00
u1,1560.50


In [109]:
quantile = spend_by_user['spend'].quantile()
quantile

np.float64(550.0)

In [111]:
spend_by_user['user_bucket']= np.where(spend_by_user['spend']<quantile, 'Bottom 50%', 'Top 50%')
spend_by_user

,spend,user_bucket
user_id,,
u2,125.25,Bottom 50%
u4,150.00,Bottom 50%
u3,950.00,Top 50%
u1,1560.50,Top 50%


In [ ]:
pd.qcut(spend_by_user['spend'], q=2, labels=['Bottom 50%', 'Top 50%'])